# 03 - Error Analysis

## Objective
Compare ERM and Group DRO predictions on the same CivilComments-WILDS test examples. The notebook studies corrected errors, newly introduced errors, shared errors, identity-specific false-positive and false-negative rates, and prediction-confidence changes.

No training or model selection is performed here.


## 1. Imports


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(); pd.set_option("display.max_columns",None); pd.set_option("display.max_rows",100); pd.set_option("display.max_colwidth",300)
print("Imports successful")


## 2. Paths and Prediction Files


In [ ]:
ERM_DIR=Path("../results/erm"); DRO_DIR=Path("../results/group_dro"); ERROR_DIR=Path("../results/error_analysis"); FIGURE_DIR=Path("../figures/results")
ERROR_DIR.mkdir(parents=True,exist_ok=True); FIGURE_DIR.mkdir(parents=True,exist_ok=True)
for p in [ERM_DIR/"test_predictions.csv",DRO_DIR/"test_predictions.csv"]:
    if not p.exists(): raise FileNotFoundError(f"Missing prediction file: {p}")
erm_df=pd.read_csv(ERM_DIR/"test_predictions.csv"); dro_df=pd.read_csv(DRO_DIR/"test_predictions.csv")
print("ERM",erm_df.shape,"Group DRO",dro_df.shape)


## 3. Align Identical Test Examples


In [ ]:
assert len(erm_df)==len(dro_df)
needed={"true_label","prediction","prob_non_toxic","prob_toxic"}
for name,df in [("ERM",erm_df),("Group DRO",dro_df)]:
    miss=needed-set(df.columns)
    if miss: raise ValueError(f"{name} missing: {sorted(miss)}")

if "uid" in erm_df.columns and "uid" in dro_df.columns:
    assert erm_df.uid.is_unique and dro_df.uid.is_unique
    cols=["uid","true_label","prediction","prob_non_toxic","prob_toxic"]
    comparison_df=erm_df.merge(dro_df[cols],on="uid",how="inner",suffixes=("_erm","_dro"),validate="one_to_one")
    assert len(comparison_df)==len(erm_df)
    assert np.array_equal(comparison_df.true_label_erm,comparison_df.true_label_dro)
    comparison_df["true_label"]=comparison_df.true_label_erm
    comparison_df=comparison_df.drop(columns=["true_label_erm","true_label_dro"])
else:
    assert np.array_equal(erm_df.true_label.values,dro_df.true_label.values)
    comparison_df=erm_df.copy().rename(columns={"prediction":"prediction_erm","prob_non_toxic":"prob_non_toxic_erm","prob_toxic":"prob_toxic_erm"})
    comparison_df["prediction_dro"]=dro_df.prediction.values
    comparison_df["prob_non_toxic_dro"]=dro_df.prob_non_toxic.values
    comparison_df["prob_toxic_dro"]=dro_df.prob_toxic.values

comparison_df=comparison_df.rename(columns={"prediction_erm":"erm_prediction","prediction_dro":"dro_prediction","prob_non_toxic_erm":"erm_prob_non_toxic","prob_non_toxic_dro":"dro_prob_non_toxic","prob_toxic_erm":"erm_prob_toxic","prob_toxic_dro":"dro_prob_toxic"})
required=["true_label","erm_prediction","dro_prediction","erm_prob_toxic","dro_prob_toxic"]
miss=[c for c in required if c not in comparison_df.columns]
if miss: raise ValueError(f"Missing comparison columns: {miss}")
print("Aligned comparison shape:",comparison_df.shape)


## 4. Outcome Categories


In [ ]:
comparison_df["erm_correct"]=comparison_df.erm_prediction==comparison_df.true_label
comparison_df["dro_correct"]=comparison_df.dro_prediction==comparison_df.true_label
conditions=[comparison_df.erm_correct&comparison_df.dro_correct,~comparison_df.erm_correct&comparison_df.dro_correct,comparison_df.erm_correct&~comparison_df.dro_correct,~comparison_df.erm_correct&~comparison_df.dro_correct]
categories=["Both correct","Group DRO fixed ERM error","Group DRO introduced error","Both wrong"]
comparison_df["outcome"]=np.select(conditions,categories,default="Unknown")
outcome_summary=comparison_df.outcome.value_counts().reindex(categories).fillna(0).astype(int).rename_axis("outcome").reset_index(name="count")
outcome_summary["percentage"]=outcome_summary["count"]/len(comparison_df)*100
outcome_summary


In [ ]:
plt.figure(figsize=(9,5)); ax=sns.barplot(data=outcome_summary,y="outcome",x="count",color="steelblue"); ax.bar_label(ax.containers[0],padding=3)
plt.xlabel("Number of test comments"); plt.ylabel(""); plt.title("ERM and Group DRO Prediction Outcomes"); plt.tight_layout()
plt.savefig(FIGURE_DIR/"prediction_outcomes.png",dpi=300,bbox_inches="tight"); plt.show()


## 5. False Positives and False Negatives


In [ ]:
comparison_df["erm_false_positive"]=(comparison_df.true_label==0)&(comparison_df.erm_prediction==1)
comparison_df["erm_false_negative"]=(comparison_df.true_label==1)&(comparison_df.erm_prediction==0)
comparison_df["dro_false_positive"]=(comparison_df.true_label==0)&(comparison_df.dro_prediction==1)
comparison_df["dro_false_negative"]=(comparison_df.true_label==1)&(comparison_df.dro_prediction==0)
error_type_summary=pd.DataFrame({"Method":["ERM","Group DRO"],"False positives":[int(comparison_df.erm_false_positive.sum()),int(comparison_df.dro_false_positive.sum())],"False negatives":[int(comparison_df.erm_false_negative.sum()),int(comparison_df.dro_false_negative.sum())]})
display(error_type_summary)
error_plot=error_type_summary.melt(id_vars="Method",var_name="Error type",value_name="Count")
plt.figure(figsize=(8,5)); sns.barplot(data=error_plot,x="Method",y="Count",hue="Error type"); plt.xlabel(""); plt.title("False Positives and False Negatives"); plt.tight_layout()
plt.savefig(FIGURE_DIR/"false_positive_negative_counts.png",dpi=300,bbox_inches="tight"); plt.show()


## 6. Errors Fixed, Introduced, and Shared


In [ ]:
dro_fixes=comparison_df.query("outcome == 'Group DRO fixed ERM error'").copy()
dro_new_errors=comparison_df.query("outcome == 'Group DRO introduced error'").copy()
both_wrong=comparison_df.query("outcome == 'Both wrong'").copy()
fixed_summary=pd.DataFrame({"Error type":["ERM false positive fixed","ERM false negative fixed"],"Count":[int(((dro_fixes.true_label==0)&(dro_fixes.erm_prediction==1)&(dro_fixes.dro_prediction==0)).sum()),int(((dro_fixes.true_label==1)&(dro_fixes.erm_prediction==0)&(dro_fixes.dro_prediction==1)).sum())]})
introduced_summary=pd.DataFrame({"Error type":["New false positive","New false negative"],"Count":[int(((dro_new_errors.true_label==0)&(dro_new_errors.dro_prediction==1)).sum()),int(((dro_new_errors.true_label==1)&(dro_new_errors.dro_prediction==0)).sum())]})
both_wrong_summary=pd.DataFrame({"Error type":["Both false positive","Both false negative"],"Count":[int(((both_wrong.true_label==0)&(both_wrong.erm_prediction==1)&(both_wrong.dro_prediction==1)).sum()),int(((both_wrong.true_label==1)&(both_wrong.erm_prediction==0)&(both_wrong.dro_prediction==0)).sum())]})
print("Fixed"); display(fixed_summary); print("Introduced"); display(introduced_summary); print("Both wrong"); display(both_wrong_summary)


## 7. Identity-Specific Error Rates


In [ ]:
IDENTITY_COLUMNS=["male","female","LGBTQ","christian","muslim","other_religions","black","white"]
missing=[i for i in IDENTITY_COLUMNS if i not in comparison_df.columns]
if missing: raise ValueError(f"Missing identity columns: {missing}")
rows=[]
for identity in IDENTITY_COLUMNS:
    mask=comparison_df[identity]>=0.5
    non_toxic=comparison_df[mask&(comparison_df.true_label==0)]
    toxic=comparison_df[mask&(comparison_df.true_label==1)]
    rows.append({"identity":identity,"n_non_toxic":len(non_toxic),"n_toxic":len(toxic),"ERM_FPR":np.nan if len(non_toxic)==0 else (non_toxic.erm_prediction==1).mean(),"GroupDRO_FPR":np.nan if len(non_toxic)==0 else (non_toxic.dro_prediction==1).mean(),"ERM_FNR":np.nan if len(toxic)==0 else (toxic.erm_prediction==0).mean(),"GroupDRO_FNR":np.nan if len(toxic)==0 else (toxic.dro_prediction==0).mean()})
identity_errors=pd.DataFrame(rows)
identity_errors["FPR_change"]=identity_errors.GroupDRO_FPR-identity_errors.ERM_FPR
identity_errors["FNR_change"]=identity_errors.GroupDRO_FNR-identity_errors.ERM_FNR
identity_errors.round(4)


## 8. Identity Error-Rate Figures


In [ ]:
fpr=identity_errors[["identity","ERM_FPR","GroupDRO_FPR"]].melt(id_vars="identity",var_name="Method",value_name="False positive rate")
fpr.Method=fpr.Method.replace({"ERM_FPR":"ERM","GroupDRO_FPR":"Group DRO"})
plt.figure(figsize=(11,5)); sns.barplot(data=fpr,x="identity",y="False positive rate",hue="Method"); plt.xticks(rotation=45); plt.title("False-Positive Rate for Non-Toxic Identity Comments"); plt.tight_layout()
plt.savefig(FIGURE_DIR/"identity_false_positive_rates.png",dpi=300,bbox_inches="tight"); plt.show()

fnr=identity_errors[["identity","ERM_FNR","GroupDRO_FNR"]].melt(id_vars="identity",var_name="Method",value_name="False negative rate")
fnr.Method=fnr.Method.replace({"ERM_FNR":"ERM","GroupDRO_FNR":"Group DRO"})
plt.figure(figsize=(11,5)); sns.barplot(data=fnr,x="identity",y="False negative rate",hue="Method"); plt.xticks(rotation=45); plt.title("False-Negative Rate for Toxic Identity Comments"); plt.tight_layout()
plt.savefig(FIGURE_DIR/"identity_false_negative_rates.png",dpi=300,bbox_inches="tight"); plt.show()


## 9. Prediction-Confidence Changes


In [ ]:
comparison_df["probability_change"]=comparison_df.dro_prob_toxic-comparison_df.erm_prob_toxic
display(comparison_df.probability_change.describe(percentiles=[0.01,0.05,0.5,0.95,0.99]))
plt.figure(figsize=(11,5)); sns.boxplot(data=comparison_df,x="probability_change",y="outcome",order=categories,showfliers=False); plt.axvline(0,color="black",linestyle="--")
plt.xlabel("Change in predicted toxicity probability (Group DRO - ERM)"); plt.ylabel(""); plt.title("Prediction Confidence Changes"); plt.tight_layout()
plt.savefig(FIGURE_DIR/"toxicity_probability_changes.png",dpi=300,bbox_inches="tight"); plt.show()


## 10. Reproducible Qualitative Samples


In [ ]:
RANDOM_SEED=42; SAMPLE_SIZE=20
fixed_sample=dro_fixes.sample(n=min(SAMPLE_SIZE,len(dro_fixes)),random_state=RANDOM_SEED)
introduced_sample=dro_new_errors.sample(n=min(SAMPLE_SIZE,len(dro_new_errors)),random_state=RANDOM_SEED)
both_wrong_sample=both_wrong.sample(n=min(SAMPLE_SIZE,len(both_wrong)),random_state=RANDOM_SEED)
review_columns=[c for c in ["uid","comment_text","true_label","erm_prediction","dro_prediction","erm_prob_toxic","dro_prob_toxic","outcome"] if c in comparison_df.columns]
print("Fixed by Group DRO"); display(fixed_sample[review_columns])
print("Introduced by Group DRO"); display(introduced_sample[review_columns])
print("Both wrong"); display(both_wrong_sample[review_columns])


## 11. Save Outputs and Print Summary


In [ ]:
outcome_summary.to_csv(ERROR_DIR/"prediction_outcome_summary.csv",index=False)
error_type_summary.to_csv(ERROR_DIR/"error_type_summary.csv",index=False)
fixed_summary.to_csv(ERROR_DIR/"fixed_error_summary.csv",index=False)
introduced_summary.to_csv(ERROR_DIR/"introduced_error_summary.csv",index=False)
both_wrong_summary.to_csv(ERROR_DIR/"both_wrong_summary.csv",index=False)
identity_errors.to_csv(ERROR_DIR/"identity_error_rates.csv",index=False)
dro_fixes.to_csv(ERROR_DIR/"dro_fixed_erm_errors.csv",index=False)
dro_new_errors.to_csv(ERROR_DIR/"dro_introduced_errors.csv",index=False)
both_wrong.to_csv(ERROR_DIR/"both_models_wrong.csv",index=False)
print("="*65); print("ERROR ANALYSIS SUMMARY"); print("="*65)
print("Total test examples:",len(comparison_df)); print("ERM errors fixed by Group DRO:",len(dro_fixes)); print("New errors introduced by Group DRO:",len(dro_new_errors)); print("Both wrong:",len(both_wrong))
print("
Largest FPR reductions:
",identity_errors[["identity","ERM_FPR","GroupDRO_FPR","FPR_change"]].sort_values("FPR_change").head().round(4).to_string(index=False))
print("
Largest FNR reductions:
",identity_errors[["identity","ERM_FNR","GroupDRO_FNR","FNR_change"]].sort_values("FNR_change").head().round(4).to_string(index=False))


## 12. Interpretation

Complete this section using the generated outputs:

- Group DRO corrected **[number]** ERM errors: **[number]** false positives and **[number]** false negatives.
- Group DRO introduced **[number]** new errors: **[number]** false positives and **[number]** false negatives.
- The greatest FPR reduction occurred for **[identity]**, from **[ERM]** to **[Group DRO]**.
- The greatest FNR reduction occurred for **[identity]**, from **[ERM]** to **[Group DRO]**.
- Both models failed on **[number]** examples. The reproducible sample suggests **[observed patterns, without overgeneralizing]**.

Interpret these findings together with `02_results_analysis.ipynb`. Do not claim universal superiority when the evidence shows a trade-off.
